# TechMind — Resumen extractivo automático

### Equipo tejONEs

#### `resumen_automatico.ipynb`

Este notebook documenta y valida la generación de resúmenes extractivos para los recursos tecnológicos que recibe TechMind. Un resumen **extractivo** no inventa ni reescribe contenido: selecciona unidades completas del documento, reduce redundancia y las devuelve en el mismo orden en que aparecían en la fuente.

## Objetivos

1. Comprobar la función reutilizable `generar_resumen(texto, n_oraciones=3, *, titulo=None)`.
2. Explicar el flujo de selección basado en TF-IDF, similitud coseno y MMR.
3. Evaluar ejemplos controlados de distintas especializaciones Tech.
4. Cargar las 1.400 filas del dataset unificado y generar un CSV enriquecido.
5. Validar extractividad, orden, límite de unidades, tratamiento de HTML, errores y latencia.
6. Mostrar cómo Backend puede sumar `resumen` a la respuesta del endpoint de carga.

## Relación con las etapas anteriores

- `05_exploracion_dataset_final.ipynb` de `feat/data-creacion-dataset-inicial` consolida el dataset con el esquema `titulo`, `texto`, `categoria`, `autor` y `tipo`.
- `06_limpieza_texto.ipynb` de `feat/data-limpieza-texto` demuestra y exporta `texto_limpio` para clasificación: elimina URLs, normaliza, convierte a minúsculas, elimina puntuación y retira stopwords.
- `07_modelo_baseline.ipynb` de `feat/data-tfidf` aplica la versión de `limpiar_texto` disponible en esa rama para entrenar un clasificador de siete categorías. Las variantes de limpieza deberán consolidarse al integrar las ramas.
- Este notebook usa la columna original `texto`, porque un resumen extractivo debe conservar redacción, mayúsculas, puntuación y términos técnicos visibles. Solo retira HTML de presentación antes de segmentar.

La categoría y el resumen son salidas independientes: el clasificador decide la especialización Tech; el resumidor selecciona las partes representativas del contenido. La implementación no se duplica en el notebook, sino que se importa desde `shared/resumen_automatico.py`, el mismo módulo previsto para Backend.

## 1. Preparación del entorno y reutilización del código

La siguiente celda importa únicamente utilidades de análisis y localiza la raíz del repositorio de forma dinámica. Esto permite ejecutar el notebook desde `data_science/notebooks/`, desde la raíz o mediante `nbconvert` sin escribir rutas absolutas del equipo.

Después se agrega la raíz a `sys.path` y se importan las funciones canónicas:

- `dividir_oraciones`: transforma texto visible en unidades extractivas estables.
- `generar_resumen`: puntúa, selecciona y ordena las unidades del resumen.
- `extraer_texto_visible`: decodifica entidades y elimina HTML de presentación sin destruir código visible.
- `contiene_html_presentacion`: ayuda a detectar HTML residual durante la auditoría.

El notebook no redefine estas funciones. Así se evita que la experimentación de Data Science y la inferencia de Backend evolucionen con implementaciones diferentes. Las stopwords españolas están incluidas en el módulo, por lo que esta etapa no descarga corpus de NLTK ni necesita acceso a internet durante la inferencia. Para reproducir el flujo y los resúmenes se requiere el entorno del proyecto, el CSV original disponible de forma local y una ejecución limpia con **Restart Kernel and Run All Cells**; las mediciones de latencia variarán entre ejecuciones.

In [1]:
import html
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


def encontrar_raiz(inicio: Path) -> Path:
    """Localiza la raíz del repositorio desde Jupyter o la terminal."""
    for candidata in [inicio, *inicio.parents]:
        if (candidata / "shared" / "resumen_automatico.py").exists():
            return candidata
    raise FileNotFoundError("No se encontró shared/resumen_automatico.py")


RAIZ_PROYECTO = encontrar_raiz(Path.cwd().resolve())
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from shared.resumen_automatico import dividir_oraciones, generar_resumen
from shared.texto_visible import (
    contiene_html_presentacion,
    extraer_texto_visible,
)

print("Módulo compartido: shared/resumen_automatico.py")

Módulo compartido: shared/resumen_automatico.py


## 2. Contrato público de `generar_resumen`

La API compartida mantiene una firma pequeña y desacoplada del clasificador:

```python
generar_resumen(
    texto: str,
    n_oraciones: int = 3,
    *,
    titulo: str | None = None,
) -> str
```

| Parámetro | Función | Reglas principales |
|---|---|---|
| `texto` | Contenido del recurso que se resumirá. | Debe ser `str`; puede incluir texto plano, entidades HTML o HTML de presentación. |
| `n_oraciones` | Cantidad máxima de unidades que se devolverán. | Debe ser un entero mayor o igual que 1; el valor predeterminado es 3. |
| `titulo` | Contexto temático opcional. | Solo modifica ligeramente la relevancia; no se incorpora como oración ni sustituye al texto. |

La salida es un `str` determinista con hasta `n_oraciones` unidades visibles y ordenadas como en la fuente. Cuando es necesario reducir el documento, se eliminan unidades duplicadas tras normalizar mayúsculas, espacios y puntuación; un documento corto puede devolverse completo aunque contenga una repetición. Un texto vacío devuelve `""`; los tipos inválidos producen `TypeError` y un límite menor que 1 produce `ValueError`. `categoria` no forma parte de la firma porque el resumen debe funcionar antes o después de la clasificación.

> En esta implementación, una *unidad* suele ser una oración, pero también puede ser un párrafo, una viñeta o un fragmento balanceado cuando el texto de origen carece de puntuación. Los umbrales de fragmentación son una política interna para evitar que un bloque OCR, índice o temario completo se convierta en un único resumen; no son parte estable del contrato público.

## 3. Enfoque técnico: TF-IDF, similitud coseno y MMR

El algoritmo no necesita entrenamiento previo ni un modelo persistido. Ajusta un vocabulario TF-IDF nuevo dentro de cada documento; no reutiliza `vectorizer.pkl` del clasificador ni el índice global de similitud de otras ramas. Que esos componentes también utilicen TF-IDF o `cosine_similarity` no significa que compartan vocabulario, artefactos o finalidad. Cada documento se procesa de manera independiente mediante el siguiente flujo:

1. **Extraer contenido visible.** Se decodifican entidades HTML, se elimina HTML de presentación y se descarta contenido no visible como `script` o `style`. Se conservan mayúsculas, puntuación y construcciones técnicas como `C++`, `C#`, `.NET`, `List<T>` o XML.
2. **Segmentar el documento.** Se reconocen signos de cierre, párrafos, listas, abreviaturas, decimales y comillas. Los bloques excesivamente largos sin puntuación se dividen en unidades balanceadas.
3. **Eliminar duplicados normalizados.** Si varias unidades conservan la misma secuencia de palabras después de normalizar mayúsculas, espacios y puntuación, se mantiene la primera aparición.
4. **Representar las unidades.** `TfidfVectorizer` calcula unigramas y bigramas, aplica frecuencia de término sublineal y excluye stopwords españolas. Las palabras frecuentes en una unidad, pero informativas dentro del documento, reciben mayor peso.
5. **Calcular relevancia.** Cada vector de unidad se compara por similitud coseno con el centroide TF-IDF del documento. Cuando hay título, la puntuación combina `0.85 × similitud con el centroide` y `0.15 × similitud con el título`.
6. **Reducir redundancia con MMR.** La primera unidad es la más relevante. Para las siguientes se maximiza `0.65 × relevancia − 0.35 × máxima similitud con las ya seleccionadas`; así se favorece información útil que aporte algo diferente.
7. **Restaurar el orden narrativo.** MMR decide qué unidades conservar, pero la salida final se ordena por su posición en el texto para mejorar la lectura.

Si TF-IDF no puede construir vocabulario —por ejemplo, cuando el documento solo contiene stopwords— se aplica un fallback determinista con las primeras unidades disponibles. Esto garantiza una respuesta estable sin inventar contenido.

> La limpieza del clasificador no se aplica directamente aquí: convertir todo a minúsculas y eliminar puntuación o stopwords antes de segmentar impediría devolver oraciones legibles y fieles a la fuente.

## 4. Diseño de los ejemplos controlados

Las siete categorías oficiales del proyecto son `Backend`, `Frontend`, `Data Science`, `Mobile`, `DevOps`, `Cloud` y `Bases de Datos`. La lista se declara en `05_exploracion_dataset_final.ipynb` de `feat/data-creacion-dataset-inicial`, y `07_modelo_baseline.ipynb` de `feat/data-tfidf` entrena y evalúa el clasificador con esas mismas clases.

Los tres documentos embebidos cubren DevOps, Data Science y Frontend, con longitudes y vocabularios diferentes. No pretenden medir el desempeño de las siete categorías ni reemplazar una evaluación etiquetada; son casos controlados para revisar el comportamiento del resumen de forma reproducible. En cada caso:

- `categoria` es metadato para presentar el ejemplo y validar coherencia con el proyecto; nunca se entrega al resumidor.
- `titulo` aporta una señal temática pequeña durante el cálculo de relevancia.
- `terminos_clave` define una comprobación sintética de cobertura para este ejemplo, no una métrica general del producto.
- `texto` contiene el documento fuente del cual deben salir todas las unidades seleccionadas.

El primer `assert` confirma que ninguna etiqueta de los ejemplos esté fuera de las siete clases oficiales. La comprensión de lista que aparece al final devuelve pares `(categoria, cantidad_de_unidades)`; sirve para inspeccionar que cada caso tenga suficientes unidades para probar una selección de tres. Ninguna de estas dos expresiones clasifica documentos ni modifica los datos.

In [2]:
CATEGORIAS = {
    "Backend", "Frontend", "Data Science", "Mobile",
    "DevOps", "Cloud", "Bases de Datos",
}

documentos = [
    {
        "categoria": "DevOps",
        "titulo": "Despliegue de contenedores con Docker y Kubernetes",
        "terminos_clave": {"docker", "kubernetes"},
        "texto": (
            "Docker empaqueta una aplicación junto con sus dependencias dentro de una imagen reproducible. "
            "El mismo contenedor puede ejecutarse en desarrollo, pruebas y producción sin cambiar su configuración base. "
            "Un registro privado almacena las imágenes aprobadas y conserva una versión para cada entrega. "
            "Kubernetes distribuye los contenedores entre los nodos disponibles del clúster. "
            "Los manifiestos declaran réplicas, límites de recursos y comprobaciones de salud. "
            "Cuando una instancia falla, el orquestador crea otra para mantener el servicio disponible. "
            "El pipeline de integración continua ejecuta pruebas antes de publicar una nueva imagen. "
            "Las métricas y los registros permiten detectar errores después del despliegue."
        ),
    },
    {
        "categoria": "Data Science",
        "titulo": "Entrenamiento y evaluación de un modelo predictivo",
        "terminos_clave": {"entrenamiento", "f1"},
        "texto": (
            "Un equipo reúne datos históricos de ventas y corrige valores ausentes antes del análisis. "
            "Python y pandas permiten explorar distribuciones, tendencias y relaciones entre variables. "
            "Las observaciones se dividen en conjuntos de entrenamiento y prueba para evitar una evaluación optimista. "
            "Un algoritmo de machine learning aprende patrones con los datos de entrenamiento. "
            "El equipo mide precisión, exhaustividad y F1 sobre ejemplos que el modelo nunca vio. "
            "El análisis de errores ayuda a decidir si se necesitan nuevas variables o más registros. "
            "Finalmente se versionan el modelo, el vectorizador y las métricas para reproducir el experimento."
        ),
    },
    {
        "categoria": "Frontend",
        "titulo": "Interfaz web accesible y adaptable con React",
        "terminos_clave": {"react", "accesibles"},
        "texto": (
            "La interfaz web utiliza HTML semántico para organizar encabezados, formularios y zonas de navegación. "
            "CSS Grid distribuye el contenido y adapta las columnas al ancho de cada pantalla. "
            "React divide la experiencia en componentes reutilizables con responsabilidades pequeñas. "
            "El estado de la aplicación se actualiza cuando la persona filtra resultados o completa un formulario. "
            "Las etiquetas accesibles permiten que lectores de pantalla describan botones y campos. "
            "La navegación por teclado ofrece una alternativa a quienes no utilizan un puntero. "
            "Las pruebas de interfaz verifican el comportamiento en teléfonos, tabletas y navegadores de escritorio. "
            "La carga diferida de módulos reduce el JavaScript inicial y mejora el tiempo de interacción. "
            "Las métricas de rendimiento ayudan a detectar regresiones antes de publicar la versión."
        ),
    },
]

assert all(documento["categoria"] in CATEGORIAS for documento in documentos)
[(documento["categoria"], len(dividir_oraciones(documento["texto"]))) for documento in documentos]

[('DevOps', 8), ('Data Science', 7), ('Frontend', 9)]

## 5. Criterios de aceptación de los ejemplos

La siguiente celda genera cada resumen y aplica controles estructurales antes de mostrarlo. La extractividad se evalúa sobre las unidades del **contenido visible con espacios normalizados**, no sobre una comparación byte a byte con el HTML crudo.

| Control | Qué comprueba | Por qué importa |
|---|---|---|
| `1 <= len(seleccionadas) <= 3` | El resultado contiene entre una y tres unidades. | Respeta el límite solicitado sin producir una respuesta vacía. |
| `es_extractivo` | Cada unidad seleccionada existe en las unidades visibles del documento. | Evita contenido inventado o reformulado. |
| `conserva_orden` | Las posiciones seleccionadas forman una subsecuencia creciente. | Mantiene la secuencia narrativa de la fuente. |
| `cobertura_tematica` | El resumen incluye los términos clave definidos para el caso. | Ofrece una señal sencilla de representatividad temática. |

Las columnas `oraciones_fuente` y `oraciones_resumen` conservan el nombre histórico de la API, pero contabilizan las unidades producidas por `dividir_oraciones`. Una tabla con valores `True` demuestra estos invariantes para los casos controlados; no equivale a una evaluación lingüística completa ni a una métrica como ROUGE contra resúmenes de referencia.

In [3]:
resultados = []
for documento in documentos:
    originales = dividir_oraciones(documento["texto"])
    resumen = generar_resumen(
        documento["texto"],
        n_oraciones=3,
        titulo=documento["titulo"],
    )
    seleccionadas = dividir_oraciones(resumen)
    posiciones = [originales.index(oracion) for oracion in seleccionadas]
    es_extractivo = all(oracion in originales for oracion in seleccionadas)
    conserva_orden = posiciones == sorted(posiciones)
    resumen_normalizado = resumen.casefold()
    cobertura_tematica = all(
        termino in resumen_normalizado
        for termino in documento["terminos_clave"]
    )

    assert 1 <= len(seleccionadas) <= 3
    assert es_extractivo
    assert conserva_orden
    assert cobertura_tematica

    resultados.append(
        {
            "categoría": documento["categoria"],
            "oraciones_fuente": len(originales),
            "oraciones_resumen": len(seleccionadas),
            "extractivo": es_extractivo,
            "orden_original": conserva_orden,
            "cobertura_tematica": cobertura_tematica,
            "resumen": resumen,
        }
    )

tabla_resultados = pd.DataFrame(resultados)
tabla_resultados

,categoría,oraciones_fuente,oraciones_resumen,extractivo,orden_original,cobertura_tematica,resumen
0,DevOps,8,3,True,True,True,Docker empaqueta una aplicación junto con sus ...
1,Data Science,7,3,True,True,True,Las observaciones se dividen en conjuntos de e...
2,Frontend,9,3,True,True,True,La interfaz web utiliza HTML semántico para or...


### 5.1 Revisión cualitativa de legibilidad

Las validaciones anteriores pueden detectar límites excedidos, contenido ajeno a la fuente, desorden o falta de los términos esperados. La siguiente celda imprime los tres resúmenes completos para revisar aspectos que no quedan representados por un booleano:

- que cada selección se entienda sin depender demasiado de una oración omitida;
- que los hechos principales estén alineados con el título;
- que no haya dos unidades prácticamente repetidas;
- que las transiciones sean aceptables para un método extractivo.

Esta inspección es deliberadamente manual. Un resumen extractivo puede conservar referencias como “estas medidas” o producir transiciones menos fluidas que un modelo generativo, aunque siga siendo fiel al documento.

In [4]:
for documento, resultado in zip(documentos, resultados):
    display(
        Markdown(
            f"### {documento['categoria']}: {documento['titulo']}\n\n"
            f"{resultado['resumen']}"
        )
    )

### DevOps: Despliegue de contenedores con Docker y Kubernetes

Docker empaqueta una aplicación junto con sus dependencias dentro de una imagen reproducible. Kubernetes distribuye los contenedores entre los nodos disponibles del clúster. Las métricas y los registros permiten detectar errores después del despliegue.

### Data Science: Entrenamiento y evaluación de un modelo predictivo

Las observaciones se dividen en conjuntos de entrenamiento y prueba para evitar una evaluación optimista. El equipo mide precisión, exhaustividad y F1 sobre ejemplos que el modelo nunca vio. El análisis de errores ayuda a decidir si se necesitan nuevas variables o más registros.

### Frontend: Interfaz web accesible y adaptable con React

La interfaz web utiliza HTML semántico para organizar encabezados, formularios y zonas de navegación. React divide la experiencia en componentes reutilizables con responsabilidades pequeñas. Las etiquetas accesibles permiten que lectores de pantalla describan botones y campos.

## 6. Contrato del dataset de entrada y salida

Esta sección aplica el resumidor al dataset consolidado por `05_exploracion_dataset_final.ipynb`. La etapa de creación combina Coursera, Microsoft Learn, OpenAlex y Stack Exchange, elimina duplicados y realiza una selección determinista de 200 recursos por cada categoría.

### Entrada

El esquema se valida de forma exacta y en este orden:

| Columna | Contenido | Uso en esta etapa |
|---|---|---|
| `titulo` | Título del recurso. | Contexto opcional para reforzar la relevancia. |
| `texto` | Contenido original del recurso. | Fuente visible de las unidades extractivas. |
| `categoria` | Una de las siete especializaciones Tech. | Auditoría del dataset; no entra al algoritmo de resumen. |
| `autor` | Autor o procedencia registrada. | Se conserva sin cambios. |
| `tipo` | Tipo de recurso. | Se conserva sin cambios. |

La lectura usa UTF-8, fuerza valores de tipo texto y desactiva la conversión automática de cadenas como `NA` a nulos. También se rechazan columnas faltantes, adicionales, desordenadas o categorías fuera del conjunto oficial.

### Salida

Se preservan las cinco columnas originales y se agregan al final:

| Columna nueva | Contrato |
|---|---|
| `resumen` | Texto visible con entre una y tres unidades extractivas; queda vacío si la fila falla. |
| `error_resumen` | Cadena vacía en éxito o `TipoDeError: mensaje` en una sola línea. |
| `latencia_resumen_ms` | Tiempo no negativo, en milisegundos, invertido en validar y resumir esa fila. |

El archivo resultante se llama `dataset_FINAL_UNIFICADO_techmind_resumen_automatico.csv`.

In [5]:
from time import perf_counter_ns

RUTA_DATASET = (
    RAIZ_PROYECTO / "data_science" / "data" / "procesados"
    / "dataset_FINAL_UNIFICADO_techmind.csv"
)
RUTA_SALIDA = RUTA_DATASET.with_name(
    "dataset_FINAL_UNIFICADO_techmind_resumen_automatico.csv"
)
COLUMNAS_ORIGINALES = ["titulo", "texto", "categoria", "autor", "tipo"]
COLUMNAS_RESUMEN = ["resumen", "error_resumen", "latencia_resumen_ms"]

if not RUTA_DATASET.is_file():
    raise FileNotFoundError(f"No se encontró el dataset original: {RUTA_DATASET}")

dataset = pd.read_csv(
    RUTA_DATASET,
    encoding="utf-8",
    dtype=str,
    keep_default_na=False,
)
columnas_recibidas = list(dataset.columns)
if columnas_recibidas != COLUMNAS_ORIGINALES:
    faltantes = [columna for columna in COLUMNAS_ORIGINALES if columna not in dataset.columns]
    extras = [columna for columna in dataset.columns if columna not in COLUMNAS_ORIGINALES]
    raise ValueError(
        "Esquema inválido del dataset. "
        f"Esperadas: {COLUMNAS_ORIGINALES}; recibidas: {columnas_recibidas}; "
        f"faltantes: {faltantes}; extras: {extras}"
    )

colisiones = set(COLUMNAS_RESUMEN).intersection(dataset.columns)
if colisiones:
    raise ValueError(f"El dataset original ya contiene: {sorted(colisiones)}")

categorias_dataset = set(dataset["categoria"].unique())
if categorias_dataset != CATEGORIAS:
    raise ValueError(
        "Categorías inválidas. "
        f"Esperadas: {sorted(CATEGORIAS)}; "
        f"recibidas: {sorted(categorias_dataset)}"
    )

print(f"Dataset cargado: {len(dataset):,} filas x {len(dataset.columns)} columnas")
print(
    "Entrada: "
    f"{Path(RUTA_DATASET).relative_to(RAIZ_PROYECTO).as_posix()}"
)
conteos_categoria = dataset["categoria"].value_counts()
resumen_dataset = pd.Series(
    {
        "filas": len(dataset),
        "columnas": len(dataset.columns),
        "categorias": len(categorias_dataset),
        "registros_por_categoria_min": int(conteos_categoria.min()),
        "registros_por_categoria_max": int(conteos_categoria.max()),
    }
)
resumen_dataset.to_frame("valor")

Dataset cargado: 1,400 filas x 5 columnas
Entrada: data_science/data/procesados/dataset_FINAL_UNIFICADO_techmind.csv


,valor
filas,1400
columnas,5
categorias,7
registros_por_categoria_min,200
registros_por_categoria_max,200


### 6.1 Procesamiento por fila y escritura segura

Cada fila se procesa de forma aislada mediante `resumir_fila`. El procedimiento elimina espacios externos, convierte un título vacío en `None` y llama a `generar_resumen` con un límite de tres unidades. Un `try/except` por registro evita que un documento problemático cancele las otras 1.399 filas: en caso de fallo se deja `resumen` vacío y se registra el tipo y mensaje en `error_resumen`.

`perf_counter_ns()` es un reloj monotónico apropiado para medir intervalos breves. La métrica incluye la validación y generación de esa fila, pero excluye la carga del CSV, la sobrecarga global de `DataFrame.apply`, la concatenación y la escritura del archivo. Por ello representa una medición local del resumidor por documento, no la latencia end-to-end del endpoint.

La ejecución es secuencial para favorecer determinismo y trazabilidad. Primero se escribe un archivo `*.tmp.csv` completo y después se reemplaza la salida definitiva. Este patrón reduce el riesgo de publicar un CSV parcial si la ejecución se interrumpe durante la escritura. Una nueva ejecución reemplaza el artefacto anterior.

In [6]:
def resumir_fila(fila: pd.Series) -> pd.Series:
    inicio = perf_counter_ns()
    resumen = ""
    error_resumen = ""

    try:
        texto = fila["texto"].strip()
        if not texto:
            raise ValueError("texto vacío o ausente")

        titulo = fila["titulo"].strip() or None
        resumen = generar_resumen(texto, n_oraciones=3, titulo=titulo)
        if not resumen.strip():
            raise ValueError("generar_resumen devolvió un resumen vacío")
    except Exception as error:
        mensaje = str(error).replace("\r", " ").replace("\n", " ").strip()
        error_resumen = f"{type(error).__name__}: {mensaje}"
        resumen = ""

    latencia_ms = round((perf_counter_ns() - inicio) / 1_000_000, 3)
    return pd.Series(
        {
            "resumen": resumen,
            "error_resumen": error_resumen,
            "latencia_resumen_ms": latencia_ms,
        }
    )


mediciones = dataset.apply(resumir_fila, axis=1)
dataset_resumido = pd.concat([dataset.copy(), mediciones], axis=1)

RUTA_TEMPORAL = RUTA_SALIDA.with_name(f"{RUTA_SALIDA.stem}.tmp.csv")
dataset_resumido.to_csv(
    RUTA_TEMPORAL,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
)
RUTA_TEMPORAL.replace(RUTA_SALIDA)
print(
    "Dataset generado: "
    f"{Path(RUTA_SALIDA).relative_to(RAIZ_PROYECTO).as_posix()}"
)

Dataset generado: data_science/data/procesados/dataset_FINAL_UNIFICADO_techmind_resumen_automatico.csv


## 7. Puertas de calidad del CSV generado

El archivo solo debe considerarse validado después de ejecutar la siguiente celda. La auditoría vuelve a leer el CSV y comprueba:

1. **Estructura:** conserva el número de filas, contiene exactamente las cinco columnas originales más las tres nuevas y no tiene encabezados duplicados.
2. **Preservación:** las columnas `titulo`, `texto`, `categoria`, `autor` y `tipo` son idénticas y mantienen el mismo orden de filas que la entrada.
3. **Resultado exclusivo:** cada registro tiene un resumen o un error, pero nunca ambos ni ninguno. El lote puede conservar errores diagnosticables; su cantidad se reporta como métrica.
4. **Latencia válida:** todos los valores pueden convertirse a número y son mayores o iguales que cero. No se fija un máximo porque depende del hardware y la carga de la máquina.
5. **Extractividad y orden:** cada unidad del resumen aparece en el contenido visible de la fuente y forma una subsecuencia en orden creciente.
6. **Límite:** todo resumen exitoso contiene entre una y tres unidades.
7. **HTML visible:** las entidades quedan decodificadas y no quedan etiquetas HTML de presentación. Fragmentos técnicos desconocidos, XML y genéricos como `List<T>` pueden conservarse porque son contenido útil.
8. **Reducción real:** para este dataset, ningún documento visible de más de 500 caracteres se devuelve completo como resumen.

Estos controles son invariantes funcionales. No sustituyen una evaluación humana a escala sobre cobertura, fluidez o utilidad para quien consume el endpoint.

In [7]:
verificacion = pd.read_csv(
    RUTA_SALIDA,
    encoding="utf-8",
    dtype=str,
    keep_default_na=False,
)

assert len(verificacion) == len(dataset)
assert list(verificacion.columns) == [*COLUMNAS_ORIGINALES, *COLUMNAS_RESUMEN]
assert not verificacion.columns.duplicated().any()
assert verificacion[COLUMNAS_ORIGINALES].equals(dataset[COLUMNAS_ORIGINALES])

latencias = pd.to_numeric(verificacion["latencia_resumen_ms"], errors="coerce")
assert latencias.notna().all() and latencias.ge(0).all()
tiene_resumen = verificacion["resumen"].str.strip().ne("")
tiene_error = verificacion["error_resumen"].str.strip().ne("")
assert (tiene_resumen ^ tiene_error).all()

def es_resumen_extractivo(fila: pd.Series) -> bool:
    if fila["error_resumen"].strip():
        return True

    originales = dividir_oraciones(fila["texto"])
    seleccionadas = dividir_oraciones(fila["resumen"])
    if not 1 <= len(seleccionadas) <= 3:
        return False

    cursor = 0
    for seleccionada in seleccionadas:
        while cursor < len(originales) and originales[cursor] != seleccionada:
            cursor += 1
        if cursor == len(originales):
            return False
        cursor += 1
    return True


assert verificacion.apply(es_resumen_extractivo, axis=1).all()

resumenes_exitosos = verificacion.loc[tiene_resumen, "resumen"]
unidades_resumen = resumenes_exitosos.map(
    lambda texto: len(dividir_oraciones(texto))
)
assert unidades_resumen.between(1, 3).all()
assert resumenes_exitosos.map(lambda texto: html.unescape(texto) == texto).all()
tiene_html_residual = resumenes_exitosos.map(contiene_html_presentacion)
assert not tiene_html_residual.any()

normalizar_visual = lambda texto: " ".join(texto.split())
textos_visibles = verificacion.loc[tiene_resumen, "texto"].map(
    extraer_texto_visible
).map(normalizar_visual)
resumenes_normalizados = resumenes_exitosos.map(normalizar_visual)
documentos_extensos = textos_visibles.str.len().gt(500)
documentos_extensos_completos = documentos_extensos & textos_visibles.eq(
    resumenes_normalizados
)
assert not documentos_extensos_completos.any()

metricas = pd.Series(
    {
        "filas": len(verificacion),
        "resúmenes_generados": int(tiene_resumen.sum()),
        "errores": int(tiene_error.sum()),
        "latencia_media_ms": round(float(latencias.mean()), 3),
        "latencia_p50_ms": round(float(latencias.quantile(0.50)), 3),
        "latencia_p95_ms": round(float(latencias.quantile(0.95)), 3),
        "latencia_max_ms": round(float(latencias.max()), 3),
        "máximo_unidades_resumen": int(unidades_resumen.max()),
        "html_residual": int(tiene_html_residual.sum()),
        "documentos_extensos_completos": int(documentos_extensos_completos.sum()),
    }
)
display(metricas.to_frame("valor"))

auditoria_por_categoria = (
    verificacion.assign(
        resumen_generado=tiene_resumen.astype(int),
        error_generado=tiene_error.astype(int),
        latencia_numerica_ms=latencias,
    )
    .groupby("categoria", sort=False)
    .agg(
        filas=("categoria", "size"),
        resumenes_generados=("resumen_generado", "sum"),
        errores=("error_generado", "sum"),
        latencia_p50_ms=("latencia_numerica_ms", "median"),
    )
)
display(auditoria_por_categoria)

,valor
filas,1400.000
resúmenes_generados,1400.000
errores,0.000
latencia_media_ms,5.592
latencia_p50_ms,5.213
latencia_p95_ms,8.590
latencia_max_ms,15.323
máximo_unidades_resumen,3.000
html_residual,0.000
documentos_extensos_completos,0.000


,filas,resumenes_generados,errores,latencia_p50_ms
categoria,,,,
Backend,200,200,0,5.4355
Bases de Datos,200,200,0,5.3715
Cloud,200,200,0,4.8340
Data Science,200,200,0,4.7950
DevOps,200,200,0,5.1840
Frontend,200,200,0,5.7240
Mobile,200,200,0,5.3240


### 7.1 Interpretación de las métricas

La tabla resume el resultado de esta ejecución concreta:

- `filas`, `resúmenes_generados` y `errores` permiten comprobar la cobertura del lote.
- `latencia_media_ms` es sensible a valores extremos; `latencia_p50_ms` representa la mediana y `latencia_p95_ms` indica el tiempo por debajo del cual terminó aproximadamente el 95 % de las filas. `latencia_max_ms` muestra el caso más lento observado.
- `máximo_unidades_resumen` debe ser menor o igual que tres.
- `html_residual` y `documentos_extensos_completos` deben permanecer en cero para este dataset.

Las latencias cambian entre equipos y ejecuciones, por lo que no deben interpretarse como un SLA de Backend. La tabla agregada por categoría permite comprobar cobertura y errores sin guardar títulos, textos, autores, resúmenes ni otros datos de registros individuales dentro del notebook.

## 8. Contrato de integración con Backend

El clasificador baseline determina `categoria`; el resumidor no la predice ni la modifica. Backend puede ejecutar ambos componentes sobre el mismo recurso y añadir `resumen: str` a su esquema Pydantic de salida.

Las ramas de Backend todavía muestran nombres de entrada diferentes. La adaptación queda en la capa del endpoint:

- contrato `texto` / `titulo`: `generar_resumen(payload.texto, titulo=payload.titulo)`;
- contrato `texto_crudo` / `titulo_documento`: `generar_resumen(datos.texto_crudo, titulo=datos.titulo_documento)`.

La carpeta `shared` debe incluirse en la imagen o paquete desplegado y Backend debe instalar una versión compatible de `scikit-learn`. Si la ruta es asíncrona y acepta documentos grandes, el equipo también debe definir un límite de tamaño y decidir si ejecuta el trabajo de CPU en un threadpool para no bloquear el event loop.

> El JSON siguiente es ilustrativo: este notebook no carga el clasificador, no llama al endpoint y no demuestra que el router ya esté integrado. La categoría se toma del ejemplo controlado, mientras que `id` y los demás campos dependerán del contrato final de Backend. `error_resumen` y `latencia_resumen_ms` pertenecen al artefacto batch y no se asumen automáticamente como campos del endpoint.

In [8]:
documento = documentos[0]
respuesta_endpoint = {
    "id": "doc-001",
    "titulo": documento["titulo"],
    "categoria": documento["categoria"],  # salida del clasificador
    "resumen": generar_resumen(
        documento["texto"],
        titulo=documento["titulo"],
    ),
}

print(json.dumps(respuesta_endpoint, ensure_ascii=False, indent=2))

{
  "id": "doc-001",
  "titulo": "Despliegue de contenedores con Docker y Kubernetes",
  "categoria": "DevOps",
  "resumen": "Docker empaqueta una aplicación junto con sus dependencias dentro de una imagen reproducible. Kubernetes distribuye los contenedores entre los nodos disponibles del clúster. Las métricas y los registros permiten detectar errores después del despliegue."
}


## 9. Resultados, limitaciones y próximos pasos

### Resultados observados

Los tres casos controlados producen resúmenes de tres unidades, fieles al contenido visible, ordenados como en la fuente y con los términos temáticos definidos para cada prueba. La diversidad de MMR y los casos límite de segmentación se cubren con mayor profundidad en `tests/test_resumen_automatico.py`.

Al ejecutarse sobre el archivo local actual, el notebook procesa las 1.400 filas y genera `dataset_FINAL_UNIFICADO_techmind_resumen_automatico.csv`. La tabla de auditoría anterior es la fuente de verdad para la cantidad de éxitos, errores y latencias de cada ejecución; las puertas de calidad exigen un máximo de tres unidades, extractividad en orden, ausencia de HTML de presentación y preservación exacta de las cinco columnas originales.

### Limitaciones

- TF-IDF mide coincidencia léxica dentro de un documento; no comprende sinónimos o relaciones semánticas como un modelo contextual.
- MMR reduce repetición, pero no garantiza que las transiciones entre unidades sean fluidas ni que desaparezcan todas las referencias anafóricas.
- Los ejemplos no incluyen resúmenes de referencia ni una evaluación humana a escala; las comprobaciones automáticas validan estructura y fidelidad, no utilidad total.
- La segmentación de HTML, listas, OCR y código usa heurísticas conservadoras. Los caracteres dañados que ya estén en el dataset fuente pueden conservarse en una salida extractiva.
- Las latencias del procesamiento batch no representan directamente el tiempo de respuesta del endpoint.

### Reproducibilidad y continuidad

Los CSV de datos están ignorados por Git, por lo que otra persona necesita el mismo archivo de entrada para reproducir el artefacto. Si cambia el dataset o algún módulo dentro de `shared`, se debe reiniciar el kernel y ejecutar todas las celdas en orden; ejecutar solo la celda de generación podría reutilizar imports almacenados en memoria. Las latencias y el hash del CSV de salida cambiarán entre ejecuciones aunque los resúmenes permanezcan deterministas para el mismo texto y entorno.

El siguiente paso de Backend es importar la función compartida, consolidar los nombres de entrada, declarar `resumen: str` en la respuesta y ejecutar una prueba de integración con el comando y entorno reales de despliegue.